In [1]:
import torch
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "deepseek-ai/deepseek-math-7b-rl"

tokenizer = AutoTokenizer.from_pretrained(
    "./deepseek_math_tokenizer"
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)



In [11]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
model.resize_token_embeddings(len(tokenizer))

Embedding(100003, 4096)

In [5]:
print("Tokenizer vocab:", len(tokenizer))
print("Model vocab:", model.get_input_embeddings().weight.shape[0])
print("PAD ID:", tokenizer.pad_token_id)
print("EOS ID:", tokenizer.eos_token_id)

Tokenizer vocab: 100003
Model vocab: 100003
PAD ID: 100002
EOS ID: 100001


In [28]:
import trl
print(trl.__file__)

/home/iztihad/venvs/ml/lib/python3.12/site-packages/trl/__init__.py


In [ ]:
# from trl import SFTTrainer
# from transformers import TrainingArguments, DataCollatorForSeq2Seq
# from unsloth import is_bfloat16_supported



# trainer = SFTTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     train_dataset = dataset,
#     dataset_text_field = "text",
#     max_seq_length = max_seq_length,
#     dataset_num_proc = 2, # Number of processors to use for processing the dataset
#     packing = False, # Can make training 5x faster for short sequences.
#     args = TrainingArguments(
#         per_device_train_batch_size = 8, # The batch size per GPU/TPU core
#         gradient_accumulation_steps = 8, # Number of steps to perform before each gradient accumulation
#         # warmup_steps = 5, # Few updates with low learning rate before actual training
#         # max_steps = 60, # Specifies the total number of training steps (batches) to run.
#         num_train_epochs=2,
#         warmup_ratio=0.3,
#         learning_rate = 2e-4,
#         fp16 = not is_bfloat16_supported(),
#         bf16 = is_bfloat16_supported(),
#         logging_steps = 1,
#         optim = "adamw_8bit", # Optimizer
#         weight_decay = 0.01,
#         lr_scheduler_type = "linear",
        
#         save_strategy="steps",
#         save_steps=100,
#         save_total_limit=5, 

#         seed = 3407,
#         output_dir = "outputs",
#         report_to = "none", # Use this for WandB etc for observability
#     ),
# )

/home/iztihad/venvs/ml/lib/python3.12/site-packages/trl/__init__.py


In [12]:
import transformers
import trl
import unsloth
import accelerate

print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("unsloth:", unsloth.__version__)
print("accelerate:", accelerate.__version__)

transformers: 4.57.6
trl: 0.24.0
unsloth: 2026.7.1
accelerate: 1.14.0


In [ ]:
# from unsloth.chat_templates import get_chat_template
# sys_prompt = """
# <problem>
# {}
# </problem>
# """
# message = sys_prompt.format("একটি বন্ধুকে পরীক্ষার প্রস্তুতির জন্য উৎসাহ দিয়ে ৩ লাইনের একটি বার্তা বাংলায় লেখো।")
# tokenizer = get_chat_template(
#     tokenizer,
#     chat_template = "llama-3.1",
# )
# FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# messages = [
#     {"role": "user", "content": message},
# ]
# inputs = tokenizer.apply_chat_template(
#     messages,
#     tokenize = True,
#     add_generation_prompt = True, # Must add for generation
#     return_tensors = "pt",
# ).to("cuda")

# outputs = model.generate(input_ids = inputs, max_new_tokens = 1024, use_cache = True,
#                          temperature = 1.5, min_p = 0.1)
# response = tokenizer.batch_decode(outputs)

In [7]:
model.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(100003, 4096)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-06)
      )
    )
    (norm): LlamaRM

In [13]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)


def generate_response(prompt, system_prompt=None):

    messages = []

    if system_prompt:
        messages.append({
            "role": "system",
            "content": system_prompt
        })

    messages.append({
        "role": "user",
        "content": prompt
    })

    # Create input IDs + attention mask
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=512,
        temperature=0.7,
        min_p=0.1,
        use_cache=True,
    )

    # Remove the input tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    # Decode only the generated response
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [14]:
while True:
    prompt = input("\nYou: ")

    if prompt.lower() in ["exit", "quit"]:
        break

    response = generate_response(prompt)

    print("\nAssistant:", response)


Assistant: The smallest number that is divisible by 6666 and only consists of the digit 2 is the number formed by writing the digit 2 repeatedly until the number is divisible by 6666.

To find the smallest such number, we need to find the smallest number of 2's that can be written to form a number divisible by 6666.

We can start by dividing 6666 by 2 to see how many 2's are needed to reach a number divisible by 6666.

6666 ÷ 2 = 3333

However, 3333 is not an integer, so we need to find the smallest integer greater than 3333. This integer is 3334.

Now, we multiply 3334 by 2 to find the smallest number made up of only the digit 2 that is divisible by 6666.

3334 × 2 = 6668

So, the smallest number made up of only the digit 2 that is divisible by 6666 is 6668.

To find out how many 2's are in this number, we simply count the number of 2's. There are 6668 digits 2 in the number.

Therefore, the smallest number of 2's that can be written to form a number divisible by 6666 is 6668.
The an

In [7]:
import trl
import trl.trainer.sft_config

print(trl.__file__)
print(trl.trainer.sft_config.__file__)

/home/iztihad/venvs/ml/lib/python3.12/site-packages/trl/__init__.py
/home/iztihad/venvs/ml/lib/python3.12/site-packages/trl/trainer/sft_config.py
